# `engine.py` Reference

`run_tournament` drives the round loop: compute standings → build context →
select pairing function → play matches → repeat.

The `PairingSpec` type lets you use one function, a list (indexed by round),
or a dict (keyed by round number, falls back to the latest key ≤ current round).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(
    Path.cwd().parent.parent
    if Path.cwd().name == 'reference'
    else Path.cwd().parent
))

from random import Random
print('ready')

In [ ]:
from tournament.generators import make_players, random_match
from tournament.engine import run_tournament
from tournament.pairing import get as get_pairing

rng = Random(42)
players = make_players(8, rng)

## Single function — reused every round

In [ ]:
t1 = run_tournament(players, n_rounds=5,
                    pairing=get_pairing('adjacent'), rng=rng)
print(f'{len(t1.rounds)} rounds, {len(t1.rounds[0].matches)} matches per round')
print('Round 1 match-ups:')
for m in t1.rounds[0].matches:
    print(f'  {players[m.player_a].name} vs {players[m.player_b].name}')

## List spec — one function per index (last entry clamped)

In [ ]:
spec_list = [get_pairing('random'), get_pairing('fold')]
t2 = run_tournament(players, n_rounds=5, pairing=spec_list, rng=Random(42))

from tournament.standings import compute_records
print('Round 1 (random) match-ups:')
for m in t2.rounds[0].matches:
    print(f'  {players[m.player_a].name} vs {players[m.player_b].name}')
print('Round 2 (fold) match-ups:')
for m in t2.rounds[1].matches:
    print(f'  {players[m.player_a].name} vs {players[m.player_b].name}')

## Dict spec — explicit per-round, fall back to latest key

In [ ]:
spec_dict = {1: get_pairing('random'), 3: get_pairing('fold')}
t3 = run_tournament(players, n_rounds=5, pairing=spec_dict, rng=Random(42))
print('Rounds 1-5 use: random, random, fold, fold, fold')
print(f'{len(t3.rounds)} rounds completed')

## Custom `match_model` — skill-blind baseline

In [ ]:
t4 = run_tournament(players, n_rounds=5,
                    pairing=get_pairing('adjacent'),
                    rng=Random(42),
                    match_model=random_match)
print(f'Skill-blind: {len(t4.rounds)} rounds')

# Skill-blind standings should be noisier
records_skilled = compute_records(t1)
records_blind   = compute_records(t4)
print(f'\n{"Name":<8} {"skilled W":>10} {"blind W":>10}')
for p in players:
    print(f'{p.name:<8} {records_skilled[p.pid].wins:>10} {records_blind[p.pid].wins:>10}')

## `_pairing_for_round` — internal resolver

In [ ]:
from tournament.engine import _pairing_for_round

fn_single = _pairing_for_round(get_pairing('adjacent'), 3)
print('Single fn, round 3:', fn_single.__name__)

fn_list = _pairing_for_round([get_pairing('random'), get_pairing('fold')], 5)
print('List spec, round 5 (clamped):', fn_list.__name__)

fn_dict = _pairing_for_round({1: get_pairing('random'), 3: get_pairing('fold')}, 2)
print('Dict spec, round 2 (fallback to key=1):', fn_dict.__name__)

fn_dict3 = _pairing_for_round({1: get_pairing('random'), 3: get_pairing('fold')}, 4)
print('Dict spec, round 4 (fallback to key=3):', fn_dict3.__name__)